## **Diagnostico de Calidad**

In [0]:
%sql
--ANALISIS DE VALORES NULL O INVALIDOS
SELECT

COUNT(*) AS total_registros,

COUNT(CASE WHEN precio IS NULL THEN 1 END) AS precio_null,
COUNT(CASE WHEN precio <=0 THEN 1 END) AS precio_negativo,
COUNT(CASE WHEN precio = 0.1 THEN 1 END) AS precio_con_01,

COUNT(CASE WHEN total IS NULL THEN 1 END) AS total_null,
COUNT(CASE WHEN total <=0 THEN 1 END) AS total_negativo,
COUNT(CASE WHEN total = 0.1 THEN 1 END) AS total_con_01,

COUNT(CASE WHEN cant IS NULL THEN 1 END) AS cant_null,
COUNT(CASE WHEN cant <=0 THEN 1 END) AS cant_negativo,

COUNT(CASE WHEN descrip IS NULL OR descrip='' THEN 1 END) AS descrip_null

FROM catalog_ventas.silver.ventas_clean

In [0]:
%sql
--ANALISIS DE DEDUPLICIDAD EN VENTAS (venta, articulo, descrip)
SELECT COUNT(*) AS total,
       COUNT(DISTINCT id_venta, articulo, descrip) AS unicos
FROM catalog_ventas.silver.ventas_clean
WHERE articulo IS NOT NULL
  AND articulo != 0
  AND descrip IS NOT NULL;

In [0]:
%sql
--DETALLE Y FRECUENCIA DE DESCRIPCIONES POR ARTICULO INCONSISTENTE
WITH base AS (
  SELECT
    articulo,
    descrip,
    COUNT(*) AS registros
  FROM catalog_ventas.silver.ventas_clean
  GROUP BY articulo, descrip
),
inconsistentes AS (
  SELECT articulo
  FROM base
  GROUP BY articulo
  HAVING COUNT(*) > 1
)

SELECT
  b.*,
  ROW_NUMBER() OVER (PARTITION BY b.articulo ORDER BY b.registros DESC) AS ranking
FROM base b
JOIN inconsistentes i
  ON b.articulo = i.articulo
ORDER BY b.articulo, ranking;

In [0]:
%sql
--ELIMINAR DUPLICADOS MANTENIENDO UN SOLO REGISTRO POR COMBINACION 
WITH deduplicados AS (
SELECT
  id_venta,
  articulo,
  descrip,
  ROW_NUMBER() OVER(PARTITION BY id_venta, articulo, descrip ORDER BY articulo DESC) AS rn
FROM catalog_ventas.silver.ventas_clean
WHERE articulo IS NOT NULL
  AND articulo !=0
  AND descrip IS NOT NULL
)
SELECT * FROM deduplicados WHERE rn = 1 